# OAC Catalog ACL Report — Gold Layer

**Version:** 1.0  
**Purpose:** Aggregate Silver `OAC_CATALOG_ACL_SILVER` into four analytics-ready summary views written to a single Gold table for OAC admin report consumption.

**Source:** `arganoadw_oacuser_sh.oacuser.OAC_CATALOG_ACL_SILVER`  
**Target:** `arganoadw_oacuser_sh.oacuser.OAC_CATALOG_ACL_GOLD`

---
### Gold Views

| VIEW_TYPE | Description | OAC Canvas |
|---|---|---|
| `PERMISSION_SUMMARY` | Item counts by catalog type, account, permission level, risk level | Canvas 1, 2 |
| `RISK_SUMMARY` | Top N highest-risk item+account combinations | Canvas 3 |
| `OWNER_SUMMARY` | Item counts and max risk score per owner | Canvas 4 |
| `ACCOUNT_COVERAGE` | How many items each principal has access to by catalog type | Canvas 5, 6 |

---
### Notes
- Gold reads exclusively from Silver — never from Bronze directly
- No `tokens.json` required — pure Spark transform
- Single table with `VIEW_TYPE` column — one OAC dataset covers all four views
- Attach Spark cluster before running

## Section 1 — Imports & Configuration
All configuration centralised here. Gold reads exclusively from Silver — never from Bronze directly. This ensures Gold always reflects enriched, quality-checked data.

- **`TOP_N_RISK`** — controls how many highest-risk items appear in `RISK_SUMMARY`. Default 50 covers the most actionable items. Increase if the catalog grows significantly.
- **`QUALITY_FILTER`** — only `OK` rows from Silver are included. Rows flagged `MISSING_PRINCIPAL` or `MISSING_PERMISSIONS` are excluded to ensure clean aggregation totals.
- **VIEW_TYPE values** written to Gold: `PERMISSION_SUMMARY`, `RISK_SUMMARY`, `OWNER_SUMMARY`, `ACCOUNT_COVERAGE`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, LongType, DoubleType
)
from datetime import datetime, timezone

# ── Source & Target ──────────────────────────────────────────
AIDP_CATALOG = 'arganoadw_oacuser_sh'
AIDP_SCHEMA  = 'oacuser'
SILVER_TABLE = 'OAC_CATALOG_ACL_SILVER'
GOLD_TABLE   = 'OAC_CATALOG_ACL_GOLD'

SILVER_FULL  = f'{AIDP_CATALOG}.{AIDP_SCHEMA}.{SILVER_TABLE}'
GOLD_FULL    = f'{AIDP_CATALOG}.{AIDP_SCHEMA}.{GOLD_TABLE}'

# ── Gold Configuration ───────────────────────────────────────
TOP_N_RISK     = 50      # Max rows in RISK_SUMMARY view
QUALITY_FILTER = 'OK'    # Only include Silver rows passing QC

print('=' * 50)
print('  SECTION 1 COMPLETE: Imports & Configuration')
print(f'  Source : {SILVER_FULL}')
print(f'  Target : {GOLD_FULL}')
print(f'  Top N Risk items : {TOP_N_RISK}')
print(f'  Quality filter   : {QUALITY_FILTER}')
print('=' * 50)

## Section 2 — Read Silver
Reads from the Silver table via the AIDP External Catalog connection. Quality filter is applied immediately so all downstream Gold aggregations work on clean rows only.

A large exclusion count (total vs filtered) warrants investigation of the Silver `DATA_QUALITY_FLAG` distribution before proceeding.

In [ ]:
spark = SparkSession.builder.appName('oac_acl_gold').getOrCreate()

df_silver_raw = spark.table(SILVER_FULL)
df_silver     = df_silver_raw.filter(F.col('DATA_QUALITY_FLAG') == QUALITY_FILTER)

total_raw      = df_silver_raw.count()
total_filtered = df_silver.count()
excluded       = total_raw - total_filtered

print('=' * 50)
print('  SECTION 2 COMPLETE: Silver Table Loaded')
print(f'  Total Silver rows : {total_raw:,}')
print(f'  Quality OK rows   : {total_filtered:,}')
print(f'  Excluded rows     : {excluded:,}')
print('=' * 50)

## Section 3 — Build Gold Views
Four aggregated DataFrames are built from Silver data. Each adds a `VIEW_TYPE` column so they can be unioned into a single Gold table while remaining filterable in OAC. `GOLD_CREATED_AT` is stamped on every row so the OAC report can show when Gold was last refreshed.

**VIEW 1 — PERMISSION_SUMMARY:** Groups by catalog type, account, permission level and risk level. Counts items at each combination. Primary dataset for Overview and Access Matrix canvases. Item owner is not grouped here — use `OWNER_SUMMARY` for ownership analysis.

**VIEW 2 — RISK_SUMMARY:** Selects the top N highest-risk individual item+account combinations, ordered by `RISK_SCORE` desc. Only includes rows where `RISK_SCORE > 0`. Feeds the High Privilege Alert canvas.

**VIEW 3 — OWNER_SUMMARY:** Groups by item owner and catalog type. `MAX_RISK_SCORE` shows the highest risk score among all items they own — a quick indicator of whether an owner has high-risk items in their catalog area.

**VIEW 4 — ACCOUNT_COVERAGE:** Groups by account and catalog type to show how many items each principal has access to. Permission flag sums show total grants per type. Feeds the stacked bar and access breadth canvases.

In [ ]:
gold_ts = datetime.now(tz=timezone.utc).isoformat()

# ── View 1: PERMISSION_SUMMARY ───────────────────────────────
df_permission_summary = (df_silver
    .groupBy(
        'CATALOG_TYPE_LABEL', 'ACCOUNT_NAME', 'ACCOUNT_CATEGORY',
        'ACCOUNT_TYPE', 'PERMISSION_LEVEL', 'RISK_LEVEL'
    )
    .agg(
        F.count('ITEM_ID').alias('ITEM_COUNT'),
        F.countDistinct('ITEM_ID').alias('DISTINCT_ITEM_COUNT'),
        F.sum('PERM_READ').alias('TOTAL_READ'),
        F.sum('PERM_WRITE').alias('TOTAL_WRITE'),
        F.sum('PERM_DELETE').alias('TOTAL_DELETE'),
        F.sum('PERM_CHANGE_PERM').alias('TOTAL_CHANGE_PERM'),
        F.sum('PERM_TAKE_OWN').alias('TOTAL_TAKE_OWN'),
        F.avg('RISK_SCORE').alias('AVG_RISK_SCORE'),
        F.max('RISK_SCORE').alias('MAX_RISK_SCORE')
    )
    .withColumn('VIEW_TYPE',       F.lit('PERMISSION_SUMMARY'))
    .withColumn('GOLD_CREATED_AT', F.lit(gold_ts))
    .withColumn('ITEM_NAME',       F.lit(None).cast(StringType()))
    .withColumn('ITEM_PATH',       F.lit(None).cast(StringType()))
    .withColumn('ITEM_OWNER',      F.lit(None).cast(StringType()))
    .withColumn('RISK_SCORE',      F.lit(None).cast(IntegerType()))
    .withColumn('ITEM_MODIFIED_TS',F.lit(None).cast(StringType()))
)

# ── View 2: RISK_SUMMARY ─────────────────────────────────────
df_risk_summary = (df_silver
    .filter(F.col('RISK_SCORE') > 0)
    .select(
        'CATALOG_TYPE_LABEL', 'ITEM_NAME', 'ITEM_PATH', 'ITEM_OWNER',
        'ACCOUNT_NAME', 'ACCOUNT_CATEGORY', 'ACCOUNT_TYPE',
        'PERMISSION_LEVEL', 'RISK_LEVEL', 'RISK_SCORE',
        F.col('ITEM_MODIFIED_TS').cast(StringType()).alias('ITEM_MODIFIED_TS')
    )
    .orderBy(F.col('RISK_SCORE').desc())
    .limit(TOP_N_RISK)
    .withColumn('VIEW_TYPE',           F.lit('RISK_SUMMARY'))
    .withColumn('GOLD_CREATED_AT',     F.lit(gold_ts))
    .withColumn('ITEM_COUNT',          F.lit(None).cast(LongType()))
    .withColumn('DISTINCT_ITEM_COUNT', F.lit(None).cast(LongType()))
    .withColumn('TOTAL_READ',          F.lit(None).cast(LongType()))
    .withColumn('TOTAL_WRITE',         F.lit(None).cast(LongType()))
    .withColumn('TOTAL_DELETE',        F.lit(None).cast(LongType()))
    .withColumn('TOTAL_CHANGE_PERM',   F.lit(None).cast(LongType()))
    .withColumn('TOTAL_TAKE_OWN',      F.lit(None).cast(LongType()))
    .withColumn('AVG_RISK_SCORE',      F.lit(None).cast(DoubleType()))
    .withColumn('MAX_RISK_SCORE',      F.lit(None).cast(IntegerType()))
)

# ── View 3: OWNER_SUMMARY ────────────────────────────────────
df_owner_summary = (df_silver
    .groupBy('ITEM_OWNER', 'CATALOG_TYPE_LABEL')
    .agg(
        F.countDistinct('ITEM_ID').alias('DISTINCT_ITEM_COUNT'),
        F.count('ITEM_ID').alias('ITEM_COUNT'),
        F.max('RISK_SCORE').alias('MAX_RISK_SCORE'),
        F.avg('RISK_SCORE').alias('AVG_RISK_SCORE'),
        F.sum('PERM_CHANGE_PERM').alias('TOTAL_CHANGE_PERM'),
        F.sum('PERM_TAKE_OWN').alias('TOTAL_TAKE_OWN')
    )
    .withColumn('VIEW_TYPE',       F.lit('OWNER_SUMMARY'))
    .withColumn('GOLD_CREATED_AT', F.lit(gold_ts))
    .withColumn('ACCOUNT_NAME',    F.lit(None).cast(StringType()))
    .withColumn('ACCOUNT_CATEGORY',F.lit(None).cast(StringType()))
    .withColumn('ACCOUNT_TYPE',    F.lit(None).cast(StringType()))
    .withColumn('PERMISSION_LEVEL',F.lit(None).cast(StringType()))
    .withColumn('RISK_LEVEL',      F.lit(None).cast(StringType()))
    .withColumn('RISK_SCORE',      F.lit(None).cast(IntegerType()))
    .withColumn('ITEM_NAME',       F.lit(None).cast(StringType()))
    .withColumn('ITEM_PATH',       F.lit(None).cast(StringType()))
    .withColumn('TOTAL_READ',      F.lit(None).cast(LongType()))
    .withColumn('TOTAL_WRITE',     F.lit(None).cast(LongType()))
    .withColumn('TOTAL_DELETE',    F.lit(None).cast(LongType()))
    .withColumn('ITEM_MODIFIED_TS',F.lit(None).cast(StringType()))
)

# ── View 4: ACCOUNT_COVERAGE ─────────────────────────────────
df_account_coverage = (df_silver
    .groupBy('ACCOUNT_NAME', 'ACCOUNT_CATEGORY', 'ACCOUNT_TYPE', 'CATALOG_TYPE_LABEL')
    .agg(
        F.countDistinct('ITEM_ID').alias('DISTINCT_ITEM_COUNT'),
        F.count('ITEM_ID').alias('ITEM_COUNT'),
        F.sum('PERM_READ').alias('TOTAL_READ'),
        F.sum('PERM_WRITE').alias('TOTAL_WRITE'),
        F.sum('PERM_DELETE').alias('TOTAL_DELETE'),
        F.sum('PERM_CHANGE_PERM').alias('TOTAL_CHANGE_PERM'),
        F.sum('PERM_TAKE_OWN').alias('TOTAL_TAKE_OWN'),
        F.max('RISK_SCORE').alias('MAX_RISK_SCORE'),
        F.avg('RISK_SCORE').alias('AVG_RISK_SCORE')
    )
    .withColumn('VIEW_TYPE',       F.lit('ACCOUNT_COVERAGE'))
    .withColumn('GOLD_CREATED_AT', F.lit(gold_ts))
    .withColumn('PERMISSION_LEVEL',F.lit(None).cast(StringType()))
    .withColumn('RISK_LEVEL',      F.lit(None).cast(StringType()))
    .withColumn('RISK_SCORE',      F.lit(None).cast(IntegerType()))
    .withColumn('ITEM_NAME',       F.lit(None).cast(StringType()))
    .withColumn('ITEM_PATH',       F.lit(None).cast(StringType()))
    .withColumn('ITEM_OWNER',      F.lit(None).cast(StringType()))
    .withColumn('ITEM_MODIFIED_TS',F.lit(None).cast(StringType()))
)

print('=' * 50)
print('  SECTION 3 COMPLETE: Gold Views Built')
print(f'  PERMISSION_SUMMARY rows : {df_permission_summary.count():,}')
print(f'  RISK_SUMMARY rows       : {df_risk_summary.count():,}')
print(f'  OWNER_SUMMARY rows      : {df_owner_summary.count():,}')
print(f'  ACCOUNT_COVERAGE rows   : {df_account_coverage.count():,}')
print('=' * 50)

## Section 4 — Union & Write Gold Table
The four Gold views are unioned into a single DataFrame before writing. A single Gold table with a `VIEW_TYPE` column is preferred over four separate tables because:
- One OAC dataset connection covers all four views
- `VIEW_TYPE` can be used as a canvas-level filter in OAC
- Simpler catalog management — one table to govern
- All views share the same snapshot timestamp (`GOLD_CREATED_AT`)

Each view fills unused columns with `NULL` (cast to correct type) so the union schema is consistent. Write strategy mirrors Bronze and Silver — full overwrite every run.

In [ ]:
FINAL_COLS = [
    'VIEW_TYPE', 'CATALOG_TYPE_LABEL', 'ITEM_NAME', 'ITEM_PATH',
    'ITEM_OWNER', 'ACCOUNT_NAME', 'ACCOUNT_CATEGORY', 'ACCOUNT_TYPE',
    'PERMISSION_LEVEL', 'RISK_LEVEL', 'RISK_SCORE',
    'ITEM_COUNT', 'DISTINCT_ITEM_COUNT',
    'TOTAL_READ', 'TOTAL_WRITE', 'TOTAL_DELETE',
    'TOTAL_CHANGE_PERM', 'TOTAL_TAKE_OWN',
    'AVG_RISK_SCORE', 'MAX_RISK_SCORE',
    'ITEM_MODIFIED_TS', 'GOLD_CREATED_AT'
]

df_gold = (df_permission_summary.select(FINAL_COLS)
    .union(df_risk_summary.select(FINAL_COLS))
    .union(df_owner_summary.select(FINAL_COLS))
    .union(df_account_coverage.select(FINAL_COLS))
)

(df_gold.write
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable(GOLD_FULL))

count = spark.table(GOLD_FULL).count()

print('=' * 50)
print('  SECTION 4 COMPLETE: Gold Table Written')
print(f'  Table   : {GOLD_FULL}')
print(f'  Rows    : {count:,}')
print(f'  Columns : {len(FINAL_COLS)}')
print('  Status  : Queryable in AIDP + OAC')
print('=' * 50)

## Section 5 — Gold Summary & Preview
Prints row counts per `VIEW_TYPE` and a sample preview of each view so the developer can validate the Gold table before pointing the OAC report at it.

**Expected output for this OAC instance:**
- `PERMISSION_SUMMARY` — small number of rows (~10–20): 2 principals × 5 catalog types
- `RISK_SUMMARY` — up to `TOP_N_RISK` rows ordered by `RISK_SCORE` desc — should show BI Service Administrator rows at top
- `OWNER_SUMMARY` — ~4 owners × 5 catalog types = ~20 rows
- `ACCOUNT_COVERAGE` — ~2 accounts × 5 catalog types = ~10 rows

In [ ]:
print('\n📊 Gold row counts by VIEW_TYPE:')
spark.table(GOLD_FULL) \
    .groupBy('VIEW_TYPE').count() \
    .orderBy('VIEW_TYPE').show()

print('📊 PERMISSION_SUMMARY preview:')
spark.table(GOLD_FULL) \
    .filter(F.col('VIEW_TYPE') == 'PERMISSION_SUMMARY') \
    .select('CATALOG_TYPE_LABEL', 'ACCOUNT_NAME', 'ACCOUNT_CATEGORY',
            'PERMISSION_LEVEL', 'RISK_LEVEL', 'ITEM_COUNT', 'MAX_RISK_SCORE') \
    .orderBy(F.col('MAX_RISK_SCORE').desc()) \
    .show(10, truncate=30)

print('📊 RISK_SUMMARY preview (top 5):')
spark.table(GOLD_FULL) \
    .filter(F.col('VIEW_TYPE') == 'RISK_SUMMARY') \
    .select('CATALOG_TYPE_LABEL', 'ITEM_NAME', 'ACCOUNT_NAME',
            'PERMISSION_LEVEL', 'RISK_LEVEL', 'RISK_SCORE') \
    .orderBy(F.col('RISK_SCORE').desc()) \
    .show(5, truncate=30)

print('📊 OWNER_SUMMARY preview:')
spark.table(GOLD_FULL) \
    .filter(F.col('VIEW_TYPE') == 'OWNER_SUMMARY') \
    .select('ITEM_OWNER', 'CATALOG_TYPE_LABEL',
            'DISTINCT_ITEM_COUNT', 'MAX_RISK_SCORE', 'TOTAL_CHANGE_PERM') \
    .orderBy(F.col('MAX_RISK_SCORE').desc()) \
    .show(10, truncate=30)

print('📊 ACCOUNT_COVERAGE preview:')
spark.table(GOLD_FULL) \
    .filter(F.col('VIEW_TYPE') == 'ACCOUNT_COVERAGE') \
    .select('ACCOUNT_NAME', 'CATALOG_TYPE_LABEL',
            'DISTINCT_ITEM_COUNT', 'TOTAL_CHANGE_PERM', 'MAX_RISK_SCORE') \
    .orderBy(F.col('TOTAL_CHANGE_PERM').desc()) \
    .show(10, truncate=30)

print('=' * 50)
print('  SECTION 5 COMPLETE: Gold Summary Printed')
print('  Review previews above before using in OAC.')
print('=' * 50)

print('\n' + '=' * 60)
print('  🏁 GOLD PIPELINE COMPLETE')
print(f'  Table: {GOLD_FULL}')
print(f'  Run time: {datetime.now(tz=timezone.utc).isoformat()}')
print('=' * 60)